In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
cpu=pd.read_csv('/content/drive/MyDrive/vm_cpu_readings.csv')
meta=pd.read_csv('/content/drive/MyDrive/vm_metadata.csv')

In [ ]:
cpu.info()

In [ ]:
meta.info()

In [ ]:
cpu['vm_id'].shape

In [ ]:
print('total unique timestamps in the cpu:',cpu['timestamp'].nunique())

In [ ]:
print('the number of null vlaues in the cpu :',cpu.isnull().sum())

In [ ]:
print('the number of duplicates vlaues in the cpu :',cpu.duplicated().sum())

In [ ]:
print('check unique VM_ID in cpu data :',cpu['vm_id'].nunique())

In [ ]:
cpu['avg_cpu'].isnull().sum()

In [ ]:
print('shape of the meta data:',meta.shape)

In [ ]:
print('check present of null values:',meta.isnull().sum())

In [ ]:
numerical=cpu[['timestamp','min_cpu','max_cpu','avg_cpu']]

In [ ]:
cpu.groupby(['timestamp'])['avg_cpu'].mean()

In [ ]:
cpu['timestamp'].unique()

In [ ]:
cpu[['avg_cpu','timestamp']].tail(5)

In [ ]:
targets=pd.DataFrame({
    'active_vm_count(t)':cpu.groupby(['timestamp'])['vm_id'].nunique(),
    'aggregate_cpu_utilization(t)': cpu.groupby(['timestamp'])['avg_cpu'].mean()
})

In [ ]:
targets

In [ ]:
targets['aggregate_cpu_utilization(t)'].unique().sum()

In [ ]:
cpu.groupby(['min_cpu','max_cpu','timestamp'])['avg_cpu'].max()

In [ ]:
print(f'the average cpu uitization at max is :{cpu['avg_cpu'].max():2f},with index number:{cpu['avg_cpu'].idxmax()}')
print(f'the average cpu uitization at min is :{cpu['avg_cpu'].min():2f},with index number:{cpu['avg_cpu'].idxmin()}')

In [ ]:
print(cpu.groupby(['timestamp'])['avg_cpu'].max())
print('*'*50)
print(cpu.groupby(['timestamp'])['avg_cpu'].min())

In [ ]:
idle=(cpu['avg_cpu']<=5).mean()*100
print(f'the Reading percentage which are idle is around:{idle:.2f}%')
maximum_utilization=(cpu['avg_cpu']>90).mean()*100
print(f'the Reading percentage which are max :{maximum_utilization:.2f}% ')
regular=(cpu['avg_cpu'].between(20,80)).mean()*100
print(f'the regular Reading are:{regular:.2f}')

In [ ]:
category=pd.cut(
    x=cpu['avg_cpu'],
    bins=[0,5,20,80,95,100],
    labels=['idle(0-5)','low(5-20)','Moderate(20-80)','HIGH(80-95)','Very_high(95-100)'],
    include_lowest=True
)
print(f'the category wise usage of the cpu are as:{category.value_counts().sort_index()}')

In [ ]:
category_counts=category.value_counts().sort_index()
plt.pie(x=category_counts,autopct='%1.1f%%',labels=category_counts.index)
plt.tight_layout()
plt.title('CPU Utilization in different Bands')

In [ ]:
total=pd.DataFrame({
    'Readings':category_counts,
    'share_%': category_counts/len(cpu)*100
})
total

In [ ]:
overall_cpu_used=cpu['avg_cpu'].mean()
overall_cpu_idel=100-overall_cpu_used
print(f'the Average CPU used is :{overall_cpu_used:.2f}\n The average CPU idle is {overall_cpu_idel:.2f}')

In [ ]:
plt.hist(x=cpu[['min_cpu','max_cpu','avg_cpu']],color=['blue','red','green'],label=['min_cpu','max_cpu','avg_cpu'],bins=10)
plt.title('CPU Reading spread')
plt.xlabel('CPU Busy levels')
plt.ylabel('CPU Readings')
plt.tight_layout()
plt.legend()
plt.show()

the most VMs are idle , can see in the pie chart . the around 1% CPU are have high usage. from histplot we can see the first (0-10) the min_cpu are most . but after that the max_cpu or we can say that the usage of the cpu is almost high from the avgerage usage and idle . this is the shift of the cpu usage we can see through it .

In [ ]:
print(targets.describe())
print(targets.isna().sum())
print(targets["active_vm_count(t)"].nunique())

In [ ]:
sns.boxplot(data=targets,y=targets['aggregate_cpu_utilization(t)'])

In [ ]:
targets["aggregate_cpu_utilization(t)"].iloc[:110].plot(figsize=(10, 4))
plt.title('how busy is an VM is over time')
plt.ylabel('CPU usage (%) ')
plt.xlabel('seconds since start')
plt.tight_layout()

# **The average CPU usage of all VMs not very much , so most CPU capacity is unused**

In [ ]:
meta['vm_category'].isna().sum()
group1=meta.groupby(['vm_category'])['avg_cpu'].agg(
    Mean='mean',Median='median',Count='count')
group1


In [ ]:
read=meta.groupby(['vm_category'])['avg_cpu'].quantile([0.10,0.25,0.50,.75,0.90,0.95]).unstack()
read.columns=['10%','25%','50%','75%','90%','95%']
print(read.round(2))

In [ ]:
read.T.plot(kind='bar', figsize=(9, 5))

plt.title('Average CPU Usage by VM Category')
plt.xlabel('Percentile')
plt.ylabel('Average CPU (%)')
plt.xticks(rotation=0)
plt.legend(title='VM Category')
plt.tight_layout()
plt.show()

In [ ]:
sns.countplot(data=meta['vm_category'])

In [ ]:
sns.boxplot(data=meta,x=meta['vm_category'],y=meta['avg_cpu'])
plt.title('CPU Usage by VM Category')
plt.show()

In [ ]:
deleted=meta['timestamp_vm_deleted']
deleted.describe()

In [ ]:
meta['timestamp_vm_created'].value_counts()

In [ ]:
meta['vm_category'].value_counts()

In [ ]:
peak=cpu['avg_cpu'].quantile(0.90)
low=cpu['avg_cpu'].quantile(0.10)
peak_target=targets['aggregate_cpu_utilization(t)'].iloc[:110].quantile(0.90)
low_target=targets['aggregate_cpu_utilization(t)'].iloc[:110].quantile(0.10)

print('the peak moment of cpu:',peak)
print('the low moment of cpu:',low)
print('the peak moment of cpu in targets :',peak_target)
print('the low moment of cpu in targets :',low_target)

In [ ]:
targets.to_csv('targets.csv')

In [ ]:
train_data=targets.iloc[:-24]
holdout=targets.iloc[-24:]

In [ ]:
print(len(train_data))
print(len(holdout))

In [ ]:
data={}
lag=6
for i in range(1,lag+1):
  column_name=f'lag{i}'
  shifted_values=train_data['aggregate_cpu_utilization(t)'].shift(i)
  data[column_name]=shifted_values

df=pd.DataFrame(data)

In [ ]:
df=df.dropna()

In [ ]:
df['y']=train_data['aggregate_cpu_utilization(t)']

In [ ]:
df

In [ ]:
x=df.drop(columns=['y']).values
y=df['y'].values

In [ ]:
print(len(x))
print(len(y))

In [ ]:
x.shape

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

In [ ]:
tcsv=TimeSeriesSplit(n_splits=3,test_size=24)

In [ ]:
!pip install optuna

In [ ]:
import optuna

In [ ]:
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from  sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error,root_mean_squared_error
from sklearn.pipeline import make_pipeline

In [ ]:
def objectives(trails):
  model_select=trails.suggest_categorical('models',['Forest_Reg','Ada_Reg','XG_Reg','SVM'])

  if model_select=='Forest_Reg':
    n_estimator=trails.suggest_int('forest',1,100)
    depth=trails.suggest_int('fores_dept',1,25)
    model=RandomForestRegressor(n_estimators=n_estimator,random_state=42,max_depth=depth)
  elif model_select=='Ada_Reg':
    n_estimator=trails.suggest_int('ada',1,100)
    learning_rate=trails.suggest_float('lr',0.001,1,log=True)
    depth=trails.suggest_int('ada_dept',1,20)
    stump=DecisionTreeRegressor(max_depth=depth)
    model=AdaBoostRegressor(estimator=stump,learning_rate=learning_rate,n_estimators=n_estimator,random_state=42)

  elif model_select=='XG_Reg':
    n_estimator=trails.suggest_int('xg',1,100)
    learning_xg=trails.suggest_float('xg_lr',0.001,1,log=True)
    model=XGBRegressor(learning_rate=learning_xg,n_estimators=n_estimator,random_state=42)
  elif model_select=='SVM':
    ker=trails.suggest_categorical('kernel',['linear', 'rbf', 'poly', 'sigmoid'])
    epsilon = trails.suggest_float(
        'svr_epsilon', 0.01, 1.0, log=True)
    C=trails.suggest_float('svr_C', 0.01, 100, log=True)
    model=make_pipeline(
        SVR(kernel=ker, epsilon=epsilon, C=C)
    )


  maes=[]
  rmse=[]

  for fit_indx,val_indx in tcsv.split(x):
    model.fit(x[fit_indx],y[fit_indx])
    pred=model.predict(x[val_indx])
    maes.append(mean_absolute_error(y[val_indx],pred))
    rmse.append(root_mean_squared_error(y[val_indx],pred))

  return np.mean(maes)

In [ ]:
from optuna.samplers import TPESampler
study=optuna.create_study(direction='minimize',sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objectives,n_trials=50)

In [ ]:
print(study.best_value)
print(study.best_params)
print(study.best_trial)

In [ ]:
df_study=(study.trials_dataframe()
      [['value','params_models']].nsmallest(20,'value')

)

In [ ]:
df_study.sort_values(ascending=True,by='value').head()

In [ ]:
from optuna.visualization import   plot_param_importances,plot_parallel_coordinate

In [ ]:
plot_parallel_coordinate(study)

In [ ]:
plot_param_importances(study)

In [ ]:
mae=[]
rmse=[]

for _,val_indx in tcsv.split(x):
  pred=x[val_indx,0]
  mae.append(mean_absolute_error(y[val_indx],pred))
  rmse.append(root_mean_squared_error(y[val_indx],pred))

print(f'the MAE of the baseline:{np.mean(mae)}')
print(f'the RMSE of the baseline:{np.mean(rmse)}')


In [ ]:
y_test = train_data['aggregate_cpu_utilization(t)'].values
seek = y_test[:86]
check = y_test[86:]

naive_base = np.repeat(seek[-1], 24)
mean_base = np.repeat(seek.mean(), 24)

x_lag= []
t_lag= []
lag = 6

for i in range(lag, len(seek)):
    x_lag.append(seek[i-lag:i])
    t_lag.append(seek[i])

model = SVR(
    kernel=study.best_params['kernel'],
    epsilon=study.best_params['svr_epsilon'],C=study.best_params['svr_C']

)
model.fit(x_lag, t_lag)

history = list(seek)
pred = []

for i in range(24):
    l = history[-6:]
    prediction = model.predict([l])[0]
    history.append(prediction)
    pred.append(prediction)

print('Naive MAE:', mean_absolute_error(check, naive_base))
print('Mean MAE:', mean_absolute_error(check, mean_base))
print('SVR MAE:', mean_absolute_error(check, pred))

In [ ]:
vm_col= 'active_vm_count(t)'
cpu_col='aggregate_cpu_utilization(t)'

train_part=targets.iloc[:110]
holdout_part=targets.iloc[110:]
print(len(holdout_part))
print(len(train_part))

preds=pd.DataFrame({
    'timestamp':holdout_part.index,
    'active_vm_count':np.repeat(train_part[vm_col].iloc[-1],24),
    'aggregate_cpu_utilization':np.repeat(train_part[cpu_col].iloc[-1],24)

})
preds.to_csv('predictions.csv', index=False)
print(preds.head(3))
print(preds.shape)

In [ ]:
h=24
series=targets['aggregate_cpu_utilization(t)'].iloc[:-h]

actual_list,forcast_list,step_list=[],[],[]

for start in range(30,len(series)-h+1):
  last_value=series.iloc[start-1]
  for i in range(1,h+1):
    actual_list.append(series.iloc[start+i-1])
    forcast_list.append(last_value)
    step_list.append(i)

# print(f'{len(actual_list)}')

actual_arr=np.array(actual_list)
forcast_arr=np.array(forcast_list)
step_arr=np.array(step_list)
error=actual_arr-forcast_arr
q1=series.quantile(0.90)
q2=series.quantile(0.10)

print(f'MAE:{round(np.abs(error).mean(),2)},\nRMSE:{round(np.sqrt((error ** 2).mean()) ,2)}')
print('Error:',error.mean())

print(f'THE peak limit:{round(q1,2)},\nTHE low limit:{round(q2,2)}')
print('margin 90%:', round(np.quantile(error, 0.90), 2))
print('margin 95%:', round(np.quantile(error, 0.95), 2))

peak,low=actual_arr>q1,actual_arr<q2
normal=(actual_arr<q1) & (actual_arr>q2)

print("Peak MAE:", round(np.abs(error[peak]).mean(), 2))
print("Low MAE:", round(np.abs(error[low]).mean(), 2))
print("Normal MAE:", round(np.abs(error[normal]).mean(), 2))